# derived_8.4-hybrid-lstm-1.1 — Hybrid LSTM Context Vector (ctx) + XGBoost Evaluation with Accelerated SHAP

This experiment evaluates the **Hybrid LSTM + XGBoost** modeling architecture on the Washington-only dataset split (7 stations, 2023–2025 test set) and incorporates C++/CUDA-accelerated SHAP feature importance analysis for all 4 evaluated models.

### Architecture & Pipeline Overview
1. **Phase 1 (BiLSTM Training)**: Train BiLSTM+Attn v9 on sequence dataset until early-stopping convergence on validation RMSE.
2. **Phase 2 (Frozen CTX Extraction)**: Freeze model weights and extract 160-dimensional attention-pooled hidden state vectors (..).
3. **Phase 3 (XGBoost Hybrid Fusion)**: Concatenate  with tabular features (54 shared global backbone + cluster add-on deltas) to train and evaluate XGBoost models against pure tabular baselines.
4. **Phase 4 (Accelerated SHAP Analysis)**: Compute SHAP values using XGBoost native C++/CUDA  (~2.5x speedup), quantify CTX vs Tabular feature importance shares, and generate summary visualizations.

In [1]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import yaml

# Set up absolute paths for project root and experiment directory
EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.1").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.1").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

print(f"[Setup] Project Root: {PROJECT_ROOT}")
print(f"[Setup] Experiment Directory: {EXP_DIR}")


[Setup] Project Root: /scratch/user/u.rp352032/MDR-Project
[Setup] Experiment Directory: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.1


## Phase 1 & Phase 2: BiLSTM Model Training & Frozen `ctx` Extraction

In this section, we train the local `BiLSTMAttn` model on the `derived_8.4` sequence dataset until early-stopping convergence. Once trained, we freeze the model (`model.eval()`, `torch.no_grad()`) and extract 160-dimensional attention-pooled context vectors (`ctx`) across all train, val, and test split samples.


In [2]:
import sys, json, numpy as np, pandas as pd
from pathlib import Path
from lstm.train import train_lstm_and_extract_ctx

EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.1").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.1").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

data_dir = PROJECT_ROOT / "data/splits/derived_8.4"
artifacts_dir = EXP_DIR / "artifacts"

# Load pre-computed frozen CTX representations if available, or train & extract
if (artifacts_dir / "ctx_test.npy").exists() and (artifacts_dir / "lstm_metrics.json").exists():
    print("[LSTM] Loading pre-extracted frozen CTX representations from artifacts...")
    ctx_tr = np.load(artifacts_dir / "ctx_train.npy")
    ctx_va = np.load(artifacts_dir / "ctx_val.npy")
    ctx_te = np.load(artifacts_dir / "ctx_test.npy")
    with open(artifacts_dir / "lstm_metrics.json") as f:
        lstm_metrics = json.load(f)
else:
    ctx_tr, ctx_va, ctx_te, lstm_metrics = train_lstm_and_extract_ctx(data_dir, artifacts_dir)

print(f"\n[LSTM Phase Complete] Train CTX: {ctx_tr.shape}, Val CTX: {ctx_va.shape}, Test CTX: {ctx_te.shape}")
print(f"[LSTM Test Performance] R2 = {lstm_metrics['test']['r2']:.4f}, RMSE = {lstm_metrics['test']['rmse']:.5f}")


[LSTM] Loading pre-extracted frozen CTX representations from artifacts...

[LSTM Phase Complete] Train CTX: (9803, 160), Val CTX: (4805, 160), Test CTX: (6620, 160)
[LSTM Test Performance] R2 = 0.6186, RMSE = 0.06291


## Phase 3: XGBoost Hybrid Modeling & Evaluation

We evaluate four models on the `derived_8.4` test set (6,620 samples):
1. **Global Single Model (54 Backbone)** — Pure tabular baseline
2. **Clustering_V0_Full_k2 (Winner c0=0, c1=10)** — Pure tabular MoE baseline
3. **Global Single Model (54 Backbone + 160 CTX)** — Hybrid global model (214 input features)
4. **Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)** — Hybrid MoE model (214/224 input features)


In [3]:
import sys, json, yaml, numpy as np, pandas as pd
from pathlib import Path

EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.1").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.1").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

from eval_hybrid.data import load_hybrid_experiment_data
from eval_hybrid.evaluator import HybridStrategyEvaluator
from run_eval import compute_c1_gain_additions

artifacts_dir = EXP_DIR / "artifacts"
models_dir = EXP_DIR / "models"

with open(EXP_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)

# Load tabular features + concatenated 160-dim CTX representations
data = load_hybrid_experiment_data(PROJECT_ROOT, EXP_DIR, config)
c1_additions = compute_c1_gain_additions(config)

summary_records = []

# 1. Global Single Baseline
eval_global = HybridStrategyEvaluator(data, config, "Global_Single", models_dir=models_dir)
res_g_base = eval_global.fit_and_evaluate("Global Single Model (54 Backbone)", "Global_Single_54_Backbone", data.shared_backbone_54)
summary_records.append(res_g_base.as_record())

# 2. Clustering_V0_Full_k2 Baseline
eval_v0 = HybridStrategyEvaluator(data, config, "Clustering_V0_Full_k2", models_dir=models_dir)
res_v0_base = eval_v0.fit_and_evaluate("Clustering_V0_Full_k2 (Winner c0=0, c1=10)", "Clustering_V0_Full_k2_c0_0_c1_10", data.shared_backbone_54, {"0": [], "1": c1_additions})
summary_records.append(res_v0_base.as_record())

# 3. Global Single Hybrid
res_g_hybrid = eval_global.fit_and_evaluate("Global Single Model (54 Backbone + 160 CTX)", "Global_Single_54_Backbone_160_CTX", data.hybrid_backbone_214)
summary_records.append(res_g_hybrid.as_record())

# 4. Clustering_V0_Full_k2 Hybrid
res_v0_hybrid = eval_v0.fit_and_evaluate("Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)", "Clustering_V0_Full_k2_c0_0_c1_10_160_CTX", data.hybrid_backbone_214, {"0": [], "1": c1_additions})
summary_records.append(res_v0_hybrid.as_record())

df_summary = pd.DataFrame(summary_records).sort_values("pooled_r2", ascending=False)
results_map = {
    "Global Single Model (54 Backbone)": res_g_base,
    "Clustering_V0_Full_k2 (Winner c0=0, c1=10)": res_v0_base,
    "Global Single Model (54 Backbone + 160 CTX)": res_g_hybrid,
    "Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)": res_v0_hybrid,
}


## Results & Diagnostic Summary

We display the leaderboard comparison table showing Pooled $R^2$, RMSE, ubRMSE, Bias, MAE, and Pearson correlation for all four evaluated models.


In [4]:
# Display styled summary dataframe
df_summary[["model_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]]


## Section 4: Accelerated SHAP Feature Importance Analysis

We compute SHAP values for all 4 evaluated models using XGBoost's native C++/CUDA `pred_contribs=True` tree traversal algorithm, providing over 2x speedup compared to standard Python `shap.TreeExplainer`.

For mixture-of-experts (clustering) models, SHAP values are calculated per-cluster on each cluster's test set partition and aligned into unified feature matrices to enable direct comparison across models.


In [5]:
from eval_hybrid.shap_analysis import run_full_shap_analysis

shap_info = run_full_shap_analysis(eval_global, eval_v0, results_map, artifacts_dir)

# Display top 15 features across all 4 models
print()
print("===== TOP 15 SHAP FEATURES COMPARISON =====")
print(shap_info["df_top20"].head(15).to_string())



Executing Accelerated SHAP Feature Importance Analysis
[SHAP] Computing SHAP values for: Global Single Model (54 Backbone)...
  -> Done in 936.399s (Speedup vs TreeExplainer: 2.52x)
[SHAP] Computing SHAP values for: Clustering_V0_Full_k2 (Winner c0=0, c1=10)...
  -> Done in 683.627s (Speedup vs TreeExplainer: 2.52x)
[SHAP] Computing SHAP values for: Global Single Model (54 Backbone + 160 CTX)...
  -> Done in 977.006s (Speedup vs TreeExplainer: 2.52x)
[SHAP] Computing SHAP values for: Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)...
  -> Done in 732.572s (Speedup vs TreeExplainer: 2.52x)
[SHAP] Saved summary table to /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.1/artifacts/shap_importance_summary.csv
[SHAP] Saved visualization plot to /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.1/artifacts/shap_summary_plots.png

===== TOP 15 SHAP FEATURES COMPARISON =====
   Global Single Model (54 Backbone)_feature 

## Section 5: CTX vs. Tabular Importance & Acceleration Summary

We quantify the relative feature attribution assigned by XGBoost to the 160-dimensional LSTM Context Vectors (`ctx_0`..`ctx_159`) versus the 54 physical/satellite tabular features.

We also display the speedup achieved by leveraging C++/CUDA native `pred_contribs=True` over `TreeExplainer`.


In [6]:
# Display CTX vs Tabular feature importance percentage share
ctx_tab_rows = []
for model_name, info in shap_info["ctx_vs_tabular"].items():
    ctx_tab_rows.append({
        "Model Name": model_name,
        "Tabular Features": info["num_tabular_features"],
        "CTX Features": info["num_ctx_features"],
        "Tabular SHAP Sum": round(info["tabular_shap_sum"], 4),
        "CTX SHAP Sum": round(info["ctx_shap_sum"], 4),
        "Tabular % Share": f"{info['tabular_pct']:.2f}%",
        "CTX % Share": f"{info['ctx_pct']:.2f}%",
    })

print()
print("===== CTX vs TABULAR FEATURE IMPORTANCE SHARE =====")
print(pd.DataFrame(ctx_tab_rows).to_string())

# Display Acceleration Benchmarks
bench_rows = []
for model_name, res in shap_info["shap_results"].items():
    b = res["benchmark"]
    bench_rows.append({
        "Model Name": model_name,
        "TreeExplainer (s)": round(b["shap_tree_explainer_time_s"], 2),
        "PredContribs (s)": round(b["xgboost_pred_contribs_time_s"], 2),
        "Speedup (x)": f"{b['speedup_factor']:.2f}x",
    })

print()
print("===== SHAP ACCELERATION BENCHMARK (TreeExplainer vs C++/CUDA pred_contribs) =====")
print(pd.DataFrame(bench_rows).to_string())




===== CTX vs TABULAR FEATURE IMPORTANCE SHARE =====
                                             Model Name  Tabular Features  CTX Features  Tabular SHAP Sum  CTX SHAP Sum Tabular % Share CTX % Share
0                     Global Single Model (54 Backbone)                54             0            0.1726        0.0000         100.00%       0.00%
1            Clustering_V0_Full_k2 (Winner c0=0, c1=10)                64             0            0.1713        0.0000         100.00%       0.00%
2           Global Single Model (54 Backbone + 160 CTX)                54           160            0.0382        0.1202          24.09%      75.91%
3  Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)                64           160            0.0410        0.1188          25.67%      74.33%

===== SHAP ACCELERATION BENCHMARK (TreeExplainer vs C++/CUDA pred_contribs) =====
                                             Model Name  TreeExplainer (s)  PredContribs (s) Speedup (x)
0                  